<a href="https://colab.research.google.com/github/maya-hoff/CS143-Portfolio/blob/main/Copy_of_F3_2_LogisticRegressionPyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS 195: Natural Language Processing
## Review of Logistic Regression and Optimization, Introduction to PyTorch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericmanley/s26-CS195NLP/blob/main/F3_2_LogisticRegressionPyTorch.ipynb)


## References

[SLP: Logistic Regression and Text Classification, Chapter 4 of Speech and Language Processing by Daniel Jurafsky & James H. Martin](https://web.stanford.edu/~jurafsky/slp3/4.pdf)


<img src="https://github.com/ericmanley/s26-CS195NLP/blob/main/images/key_disable.png?raw=1" />

In [2]:
#import sys
#!{sys.executable} -m pip install datasets transformers torch

## Review: Integer Encoding

We tried machine learning with text where each word was assigned a number - integer encoding

We have to make sure each input has the same size. Since text inputs are different sizes
* pad small ones with zeros
* truncate long ones

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer
from sklearn.metrics import accuracy_score
from sklearn import tree

dt = tree.DecisionTreeClassifier(random_state=41)

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM3-3B")

data = load_dataset("Deysi/spam-detection-dataset")


train_encoding = tokenizer(list(data["train"]["text"]),truncation=True,padding="max_length",max_length=512)
train_labels = data["train"]["label"]
test_encoding = tokenizer(list(data["test"]["text"]),truncation=True,padding="max_length",max_length=512)
test_labels = data["test"]["label"]


dt.fit(train_encoding["input_ids"],train_labels)

predictions = dt.predict(test_encoding["input_ids"])

print( accuracy_score(test_labels,predictions) )

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

data/train-00000-of-00001-daf190ce720b3d(…):   0%|          | 0.00/1.92M [00:00<?, ?B/s]

data/test-00000-of-00001-fa9b3e8ade89a33(…):   0%|          | 0.00/663k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8175 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2725 [00:00<?, ? examples/s]

0.8928440366972477


## Review: Bag-of-Words Encoding

Choose vocabulary (say 5000 most common words) one column for each word

row contains counts for each word

**Example**

*Sentence 1:* "The cat sat on the hat"

*Sentence 2:* "The dog ate the cat and the hat"

*Vocabulary:* { the, cat, sat, on, hat, dog, ate, and }


|            | the | cat | sat | on | hat | dog | ate | and |
|------------|-----|-----|-----|----|-----|-----|-----|-----|
| Sentence 1 | 2   | 1   | 1   | 1  | 1   | 0   | 0   | 0   |
| Sentence 2 | 3   | 1   | 0   | 0  | 1   | 1   | 1   | 1   |


**The downside:** this doesn't maintain any information about word order - thus the "bag" of words

`scikit-learn` provides a Bag-of-Words encoder called `CountVectorizer`


In [4]:
from sklearn import tree
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import CountVectorizer

data = load_dataset("Deysi/spam-detection-dataset")

train_texts = data["train"]["text"]
train_labels = data["train"]["label"]
test_texts = data["test"]["text"]
test_labels = data["test"]["label"]

# Consider top 5000 frequent words
# remove stop words
vectorizer = CountVectorizer(max_features=5000,stop_words="english")
vectorizer.fit(train_texts)

train_vectors = vectorizer.transform(train_texts)
test_vectors = vectorizer.transform(test_texts)

dt = tree.DecisionTreeClassifier(random_state=41)
dt.fit(train_vectors,train_labels)

predictions = dt.predict(test_vectors)

print( accuracy_score(test_labels,predictions) )

0.9724770642201835


## TD-IDF Encoding

**TF-IDF:** Term Frequency - Inverse Document Frequency

**Term Frequency:** How often does the word appear in the example, like CountVectorizer
* actually take the $\log$ of it

**Document Frequency:** What fraction of the *documents* (or *training-examples*) does the word appear in?

**Inverse Document Frequency:** (number of documents) / (number of documents containing the word)
* *I forgot to mention this last time: we take the log of this too* - measuring in orders of magnitude helps keep this from dominating the product
* if a word is in only a few documents, you get a big number
* if a word appears in lots of documents, you get a small number

When encoding a new example, multiply the Term Frequency of the word in this example by the Inverse Document Frequency of the training set
* gives higher weight to words that are differentiators
* stop words should automatically be de-emphasized

**Example:**
Document collection: all of Shakespeare's plays

The word `Romeo` appears 113 times but only in 1 document

The word `action` appears 113 time but in 31 documents

so Romeo will get a much higher weight


In [5]:
from sklearn import tree
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer

data = load_dataset("Deysi/spam-detection-dataset")

train_texts = data["train"]["text"]
train_labels = data["train"]["label"]
test_texts = data["test"]["text"]
test_labels = data["test"]["label"]

# Consider top 5000 frequent words
vectorizer = TfidfVectorizer(max_features=5000)
vectorizer.fit(train_texts)

train_vectors = vectorizer.transform(train_texts)
test_vectors = vectorizer.transform(test_texts)

dt = tree.DecisionTreeClassifier(random_state=41)
dt.fit(train_vectors,train_labels)

predictions = dt.predict(test_vectors)

print( accuracy_score(test_labels,predictions) )

0.9853211009174312


In [6]:
type(train_vectors)

scipy.sparse._csr.csr_matrix

In [7]:
type(test_labels)

datasets.arrow_dataset.Column

## PyTorch introduction

PyTorch is the go-to library for neural network models in Python

We're going to start with a simple model - we'll review Logistic Regression and see how it works in PyTorch

Let's start with the setup. First the libraries we need and loading the dataset as before

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.feature_extraction.text import TfidfVectorizer
from datasets import load_dataset


data = load_dataset("Deysi/spam-detection-dataset")

train_texts = data["train"]["text"]
test_texts  = data["test"]["text"]

vectorizer = TfidfVectorizer(max_features=5000)
train_vectors = vectorizer.fit_transform(train_texts)
test_vectors  = vectorizer.transform(test_texts)

but we need numerical values for the labels rather than strings like sklearn is ok with

In [9]:
train_labels = [1 if y == "spam" else 0 for y in data["train"]["label"]]
test_labels  = [1 if y == "spam" else 0 for y in data["test"]["label"]]

### Tensors

PyTorch has its own data structure, `tensor`, for storing arrays/vectors.

The sklearn vectorizer returns a `scipy` based matrix that has to be converted into a `numpy` array before it can be converted into a `tensor`

The label lists we made can be converted directly, but we need to `unsqueeze` it by one dimension, meaning turn a 1 dimensional tensor into a 2-dimensional one - it's like putting a list into another list where it is the only element

In [10]:
type(train_vectors)

scipy.sparse._csr.csr_matrix

In [11]:
X_train = torch.tensor(train_vectors.toarray(), dtype=torch.float32)
X_test = torch.tensor(test_vectors.toarray(), dtype=torch.float32)
X_train

tensor([[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.1507, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]])

In [12]:
y_train = torch.tensor(train_labels, dtype=torch.float32).unsqueeze(1)
y_test = torch.tensor(test_labels, dtype=torch.float32).unsqueeze(1)
y_train

tensor([[0.],
        [1.],
        [1.],
        ...,
        [0.],
        [1.],
        [1.]])

## Logistic Regression

Critical assumption with logistic regression: the real phenomenon can be modeled by this linear function

$$y = w_nx_n+w_{n-1}x_{n-1}+\cdots+w_1x_1+b$$

The $x$s: numeric inputs from Bag of Words or whatever encoding you're using

The $w$s and $b$: some weights/parameters that we learn

Positive $y$: predict True

Negative $y$: predict False

### Vector representation

This equation is often described as vectors, with the parameter vector $\boldsymbol{w} = [w_1, w_2, \ldots, w_n]$ and input vector $\boldsymbol{x} = [x_1, x_2, \ldots, x_n]$ and we use the dot product

$$y = \boldsymbol{w} \cdot \boldsymbol{x} + \boldsymbol{b}$$

### Squashing Function

We usually take the output and put it through a *squashing*/*activation* functino like the **sigmoid** function $\sigma(y) = 1/(1+e^{-y})$

<center><img src = "images/sigmoid.png" /></center>

image credit: [SLP Fig 4.1](https://web.stanford.edu/~jurafsky/slp3/4.pdf)

**Why?**
* takes big numbers and forces them to be between 0 and 1
* very close to 0 and 1 most of the time, but it looks linear near the center when we're uncertain
* differentiable, so it works with all the calculus


## Linear Model representation in PyTorch

What does this model looks like in PyTorch?
* Remember our TF-IDF vectors were set to use 5000 tokens
* We have only a single linear node

In [13]:
model = torch.nn.Linear(5000, 1)

### Discussion Question

This works because we only have a single binary answer

How does a linear separator work if you have 3 or more classes to separate?

What do you think you'd do differently in the PyTorch code?

## Learning with Logistic Regression

To learn the right $y = \boldsymbol{w} \cdot \boldsymbol{x} + \boldsymbol{b}$, we use some kind of optimization algorithm

Example: Gradient Descent
* Quick explanation and visualization: https://www.youtube.com/watch?v=qg4PchTECck
* More visualization: https://www.youtube.com/embed/GkB4vW16QHI?si=7BpkMgIqIaLXM89-&amp;start=126


### Mental hang-up

Gradient Descent is often shown as trying to find the point on a surface that is smallest, but we're **not** trying to find the minimum $\boldsymbol{x}$ that minimizes $y = \boldsymbol{w} \cdot \boldsymbol{x} + \boldsymbol{b}$.

We're trying to find a $\boldsymbol{w}$ and $\boldsymbol{b}$ that minimizes how different our guess/model $$\hat{y} = \boldsymbol{w} \cdot \boldsymbol{x} + \boldsymbol{b}$$ is from the real $y$ given all the training examples $\boldsymbol{x}$ that we have.

But the same algorithm still works!

### Loss functions

The way you measure how different your model $\hat{y}$ is from reality $y$ is called the **loss function**
* there are many you could choose

For categorical data, we usually use the **cross-entropy loss** (also called **negative log likelihood loss**) which looks like this

$$L_{CE} = -[y\log(\hat{y})+(1-y)\log(1-\hat{y})]$$

**Where does this come from?**

Let's say we see a given input $x$ - remember this is like the bag-of-words representation (or some other encoding) of some text

and we're trying to find the weights that give us the correct label $y$ (i.e., spam or not-spam)

The *probability* that $y$ is the right label given that we've observed input $x$ is

$$p(y|x) = \hat{y}^y{(1-\hat{y})}^{(1-y)}$$

Why?

* if $y=1$, then $\hat{y}^y{(1-\hat{y})}^{(1-y)}$ simplifies to $\hat{y}$
  * $p(1|x) = \hat{y}$ if the model guessed $\hat{y}=1$, then we got it right - a high probability
  * $p(1|x) = \hat{y}$ if the model guessed $\hat{y}=0$, then we got it wrong - a low probability

    
* if $y=0$, then $\hat{y}^y{(1-\hat{y})}^{(1-y)}$ simplifies to $1-\hat{y}$
  * $p(0|x) = 1-\hat{y}$ if the model guessed $\hat{y}=0$, then we got it right - a high probability
  * $p(0|x) = 1-\hat{y}$ if the model guessed $\hat{y}=1$, then we got it wrong - a low probability

Takeaway:
* when the model assigns a guess with high confidence ($\hat{y}$ close to 0 or 1) and is correct, the $p(y|x)$ expression is high
* when the model assigns a guess with high confidence ($\hat{y}$ close to 0 or 1) and is wrong, the $p(y|x)$ expression is low

#### However

the cross-entropy loss function actually uses the negative log of this $$-\log(p(y|x)) = -[y\log(\hat{y})+(1-y)\log(1-\hat{y})]$$

Why?
* turns products into sums - repeated products less than 1 get small quickly and are hard to work with
* minimize instead of maximize - smaller loss is easier to think of as being more correct, just like with mean-squared-error, you want to make it small
* when used to make changes to the weights (the *gradients*), big errors (confident and wrong) cause significantly bigger changes than small errors




### So how much do we change the weight based on this loss function?

Think of moving along each dimension independently, and we get the direction and magnitude that each weight needs to move by taking the partial derivative with respect to that dimension:

$$\frac{\partial L_{CE}(\hat{y},{y})}{\partial w_j} = (\text{do some calculus}) = (\hat{y}-y)x_j$$

The partial derivative calculates the slope along that dimension as we move towards the minimum loss - if the slope is big then we want to nudge the weight a lot in that direction, if it is small, we only want to nudge a little

**do some calculus** means plug in $y$ (right answer from training set) and $\hat{y}$ (what the model guessed) to the loss function, take the derivative with respect to this weight, and after applying the chain rule, you get $(\hat{y}-y)x_j$

This value, $(\hat{y}-y)x_j$ is called the **gradient**

except, we can also control how quickly allow the weights to change by multiplying by a small *learning rate* $\eta$

so

the new weight is $$w_j \leftarrow w_j - \eta(\hat{y}-y)x_j$$

## Loss Function and Optimizer Algorithm in PyTorch

`BCEWithLogitsLoss` is PyTorch's implementation of cross-entropy loss *when you have only two possible classes*, i.e., Binary Cross Entropy
* *logits* means the score of $y = \boldsymbol{w} \cdot \boldsymbol{x} + \boldsymbol{b}$ before squashing with the sigmoid function. This loss function will apply the sigmoid function itself, so it wants you to *input* the logits rather than the already-squashed version

`SGD` is PyTorch's implementation of gradient descent that uses this update rule
* `lr` is the learning rate, $\eta$ that we set

In [14]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

### Discussion Question

What if you have more than two categories (think an emotions dataset rather than spam/not-spam

Do some searching and see if you can find out what you're supposed to use in this case instead

## PyTorch Training Loop

In [15]:
for epoch in range(10):
    # clear stored gradients
    optimizer.zero_grad()

    # get predictions on the training examples
    logits = model(X_train)

    # calculate the loss - how wrong are the predictions?
    loss = loss_fn(logits, y_train)

    # calculate all the gradients for each parameter
    loss.backward()

    # update weights using those gradients
    optimizer.step()

    print(f"Epoch {epoch+1}, loss = {loss.item():.4f}")

Epoch 1, loss = 0.6930
Epoch 2, loss = 0.6924
Epoch 3, loss = 0.6918
Epoch 4, loss = 0.6912
Epoch 5, loss = 0.6907
Epoch 6, loss = 0.6901
Epoch 7, loss = 0.6895
Epoch 8, loss = 0.6889
Epoch 9, loss = 0.6884
Epoch 10, loss = 0.6878


Let's break this down

each parameter `param` in the model has
* `param.data` the current weight value
* `param.grad` the accumulated gradient

`optimizer.zero_grad()` zeros `param.grad` for all parameters

`loss.backward()` accumulates new values for each `param.grad` based on the loss just calculated

`optimizer.step()` updates `param.data` using `param.grad`

We could wait to do `optimizer.step()` until after doing more accumulations - this is the difference between different variations of gradient descent, like stochastic gradient descent and batch gradient descent


## Evaluation Loop

When doing evaluations, you can run `torch.no_grad()` to tell PyTorch not to track any gradients inside this block of code


In [16]:
with torch.no_grad():
    predictions = torch.sigmoid(model(X_test))
    predicted_labels = (predictions > 0.5).int()
    accuracy = (predicted_labels == y_test.int()).float().mean()
    print(accuracy)

print("Accuracy:", accuracy.item())

tensor(0.8371)
Accuracy: 0.8370642066001892


### Group Investigation

Does the accuracy improve with additional runs of the training loop?

print out `predictions` and `predictions>0.5` and `(predictions > 0.5).int()` what's going on here?

print out `y_test` and `predicted_labels == y_test.int()` what's going on here?

What happens if I print `accuracy` instead of `accuracy.item()`?

## Putting it all together

Here's the full PyTorch workflow for this problem

In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.feature_extraction.text import TfidfVectorizer
from datasets import load_dataset


data = load_dataset("Deysi/spam-detection-dataset")

train_texts = data["train"]["text"]
test_texts  = data["test"]["text"]

vectorizer = TfidfVectorizer(max_features=5000)
train_vectors = vectorizer.fit_transform(train_texts)
test_vectors  = vectorizer.transform(test_texts)

train_labels = [1 if y == "spam" else 0 for y in data["train"]["label"]]
test_labels  = [1 if y == "spam" else 0 for y in data["test"]["label"]]

X_train = torch.tensor(train_vectors.toarray(), dtype=torch.float32)
X_test = torch.tensor(test_vectors.toarray(), dtype=torch.float32)

y_train = torch.tensor(train_labels, dtype=torch.float32).unsqueeze(1)
y_test = torch.tensor(test_labels, dtype=torch.float32).unsqueeze(1)

model = torch.nn.Linear(5000, 1)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

for epoch in range(100):
    optimizer.zero_grad()

    logits = model(X_train)
    loss = loss_fn(logits, y_train)

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, loss = {loss.item():.4f}")


with torch.no_grad():
    predictions = torch.sigmoid(model(X_test))
    predicted_labels = (predictions > 0.5).int()
    accuracy = (predicted_labels == y_test.int()).float().mean()
    print(accuracy)

print("Accuracy:", accuracy.item())

Epoch 1, loss = 0.6933
Epoch 2, loss = 0.6927
Epoch 3, loss = 0.6921
Epoch 4, loss = 0.6915
Epoch 5, loss = 0.6909
Epoch 6, loss = 0.6903
Epoch 7, loss = 0.6898
Epoch 8, loss = 0.6892
Epoch 9, loss = 0.6886
Epoch 10, loss = 0.6881
Epoch 11, loss = 0.6875
Epoch 12, loss = 0.6869
Epoch 13, loss = 0.6863
Epoch 14, loss = 0.6858
Epoch 15, loss = 0.6852
Epoch 16, loss = 0.6847
Epoch 17, loss = 0.6841
Epoch 18, loss = 0.6835
Epoch 19, loss = 0.6830
Epoch 20, loss = 0.6824
Epoch 21, loss = 0.6819
Epoch 22, loss = 0.6813
Epoch 23, loss = 0.6807
Epoch 24, loss = 0.6802
Epoch 25, loss = 0.6796
Epoch 26, loss = 0.6791
Epoch 27, loss = 0.6785
Epoch 28, loss = 0.6780
Epoch 29, loss = 0.6774
Epoch 30, loss = 0.6769
Epoch 31, loss = 0.6763
Epoch 32, loss = 0.6758
Epoch 33, loss = 0.6753
Epoch 34, loss = 0.6747
Epoch 35, loss = 0.6742
Epoch 36, loss = 0.6736
Epoch 37, loss = 0.6731
Epoch 38, loss = 0.6725
Epoch 39, loss = 0.6720
Epoch 40, loss = 0.6715
Epoch 41, loss = 0.6709
Epoch 42, loss = 0.6704
E

## Applied Exploration

Select another Hugging Face dataset for text classification and get it working with this code.

Note, that if you use a dataset with more than 2 classes (a laudable goal to get working)
* You will need to change the `model` to match
* You need to find the non-binary version of cross-entropy for the loss function
* You may need format for the labels
* When evaluating, you will need to look at what the predictions look like and compute accordingly

Give a short write-up on the following
* Describe your dataset, including the distribution of the target variable
* Interpret the results - How did this dataset compare with the spam dataset? Why do you think you got the results that you did?

In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.feature_extraction.text import TfidfVectorizer
from datasets import load_dataset

loss_fn = nn.CrossEntropyLoss()

data = load_dataset("dair-ai/emotion")

train_texts = data["train"]["text"]
test_texts  = data["test"]["text"]

vectorizer = TfidfVectorizer(max_features=5000)
train_vectors = vectorizer.fit_transform(train_texts)
test_vectors  = vectorizer.transform(test_texts)

train_labels = data["train"]["label"]
test_labels  = data["test"]["label"]

X_train = torch.tensor(train_vectors.toarray(), dtype=torch.float32)
X_test = torch.tensor(test_vectors.toarray(), dtype=torch.float32)

y_train = torch.tensor(train_labels, dtype=torch.long)
y_test = torch.tensor(test_labels, dtype=torch.long)

model = torch.nn.Linear(5000, 6)
optimizer = optim.SGD(model.parameters(), lr=0.1)

for epoch in range(100):
    optimizer.zero_grad()

    logits = model(X_train)
    loss = loss_fn(logits, y_train)

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, loss = {loss.item():.4f}")


with torch.no_grad():
    logits = model(X_test)
    predicted_labels = torch.argmax(logits, dim=1)

    accuracy = (predicted_labels == y_test).float().mean()
    print("Accuracy:", accuracy.item())

print("Accuracy:", accuracy.item())

Epoch 1, loss = 1.7925
Epoch 2, loss = 1.7851
Epoch 3, loss = 1.7780
Epoch 4, loss = 1.7711
Epoch 5, loss = 1.7644
Epoch 6, loss = 1.7580
Epoch 7, loss = 1.7518
Epoch 8, loss = 1.7458
Epoch 9, loss = 1.7401
Epoch 10, loss = 1.7345
Epoch 11, loss = 1.7291
Epoch 12, loss = 1.7240
Epoch 13, loss = 1.7190
Epoch 14, loss = 1.7141
Epoch 15, loss = 1.7095
Epoch 16, loss = 1.7050
Epoch 17, loss = 1.7007
Epoch 18, loss = 1.6965
Epoch 19, loss = 1.6925
Epoch 20, loss = 1.6886
Epoch 21, loss = 1.6848
Epoch 22, loss = 1.6812
Epoch 23, loss = 1.6777
Epoch 24, loss = 1.6744
Epoch 25, loss = 1.6711
Epoch 26, loss = 1.6680
Epoch 27, loss = 1.6650
Epoch 28, loss = 1.6621
Epoch 29, loss = 1.6593
Epoch 30, loss = 1.6566
Epoch 31, loss = 1.6540
Epoch 32, loss = 1.6515
Epoch 33, loss = 1.6490
Epoch 34, loss = 1.6467
Epoch 35, loss = 1.6444
Epoch 36, loss = 1.6422
Epoch 37, loss = 1.6401
Epoch 38, loss = 1.6381
Epoch 39, loss = 1.6361
Epoch 40, loss = 1.6342
Epoch 41, loss = 1.6324
Epoch 42, loss = 1.6306
E

1. I used the dataset `dair-ai/emotion` which is a multi-class text classification dataset. It has short text snippets labeled with one of six emotions. It is split into training, validation, and test sets. The target variable is not 100% balanced, as some emotions show up more commonly than others. This imbalance may make it harder for the model to learn the less frequent emotions.

2. Compared to the spam dataset, this task is more difficult because it involves six classes instead of two. The model has to distinguish between similar emotions, which increases confusion and lowers accuracy. In contrast, the spam dataset is easier because it often contains clear keyword patterns that models can learn more easily and also has fewer categories to choose from.